# Stage 1: Mask R-CNN + ResNet50 untuk Segmentasi Optic Disc & Hard Exudate

**Pipeline:**
1. Load gambar fundus retina original dari IDRiD
2. Load ground truth mask OD / HE
3. Online augmentation dengan Albumentations
4. Train Mask R-CNN + ResNet50 (torchvision)
5. Evaluasi dengan IoU

**Referensi baseline:**
- OD: IoU = 0.843 (Suardika et al., IBIOMED 2022)
- HE: IoU = 0.993 (Maysanjaya et al., ICITRI 2022)

## 1. Import & Konfigurasi

In [27]:
import torch
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import torchvision.transforms.v2 as T

import numpy as np
import cv2
import albumentations as A
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import time
import json

print(f"PyTorch       : {torch.__version__}")
print(f"TorchVision   : {torchvision.__version__}")
print(f"MPS available : {torch.backends.mps.is_available()}")

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device        : {DEVICE}")

PyTorch       : 2.10.0
TorchVision   : 0.25.0
MPS available : True
Device        : mps


## 2. Path Dataset

In [28]:
BASE_DIR = Path(".")

ORIG_TRAIN = BASE_DIR / "1. Original Images" / "a. Training Set"
ORIG_TEST  = BASE_DIR / "1. Original Images" / "b. Testing Set"

GT_BASE_TRAIN = BASE_DIR / "2. All Segmentation Groundtruths" / "a. Training Set"
GT_BASE_TEST  = BASE_DIR / "2. All Segmentation Groundtruths" / "b. Testing Set"

# Ground truth paths
GT_PATHS = {
    "OD": {
        "train": GT_BASE_TRAIN / "5. Optic Disc",
        "test":  GT_BASE_TEST  / "5. Optic Disc",
        "suffix": "_OD.tif",
    },
    "HE": {
        "train": GT_BASE_TRAIN / "3. Hard Exudates",
        "test":  GT_BASE_TEST  / "3. Hard Exudates",
        "suffix": "_EX.tif",  # IDRiD uses _EX suffix for Hard Exudates
    },
}

print("Path konfigurasi OK")
for target, paths in GT_PATHS.items():
    train_count = len(list(paths["train"].glob("*.tif")))
    test_count = len(list(paths["test"].glob("*.tif")))
    print(f"  {target}: {train_count} train, {test_count} test")

Path konfigurasi OK
  OD: 54 train, 27 test
  HE: 54 train, 27 test


## 3. Dataset Class

In [ ]:
class IDRiDSegDataset(Dataset):
    """
    Dataset IDRiD untuk Mask R-CNN.
    Setiap gambar menghasilkan target berupa bounding box + mask untuk instance segmentation.
    """

    def __init__(self, img_dir, gt_dir, gt_suffix, transform=None, img_size=512, min_area=10):
        self.img_dir = Path(img_dir)
        self.gt_dir = Path(gt_dir)
        self.gt_suffix = gt_suffix
        self.transform = transform
        self.img_size = img_size
        self.min_area = min_area

        # Kumpulkan semua gambar yang punya ground truth
        self.samples = []
        for img_path in sorted(self.img_dir.glob("IDRiD_*.jpg")):
            stem = img_path.stem
            gt_path = self.gt_dir / f"{stem}{self.gt_suffix}"
            # Semua gambar dimasukkan, termasuk yang tanpa GT (negative samples)
            self.samples.append({
                "img_path": img_path,
                "gt_path": gt_path if gt_path.exists() else None,
                "stem": stem,
            })

        print(f"  Dataset: {len(self.samples)} gambar, "
              f"{sum(1 for s in self.samples if s['gt_path'] is not None)} dengan GT")

    def __len__(self):
        return len(self.samples)

    def _load_mask(self, gt_path):
        """Load ground truth TIF dan konversi ke binary mask."""
        img = cv2.imread(str(gt_path), cv2.IMREAD_UNCHANGED)
        if img is None:
            return None
        if img.ndim == 3:
            img = img[:, :, :3].max(axis=2)
        binary = (img > 10).astype(np.uint8)
        return binary

    def _extract_instances(self, mask):
        """
        Extract individual instances dari binary mask.
        Setiap connected component = 1 instance.
        Return: list of (instance_mask, bbox)
        """
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        instances = []
        for contour in contours:
            area = cv2.contourArea(contour)
            if area < self.min_area:
                continue
            # Buat instance mask
            inst_mask = np.zeros_like(mask)
            cv2.drawContours(inst_mask, [contour], -1, 1, cv2.FILLED)

            # Bounding box [x_min, y_min, x_max, y_max]
            x, y, w, h = cv2.boundingRect(contour)
            bbox = [x, y, x + w, y + h]

            instances.append((inst_mask, bbox))
        return instances

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Load image
        image_bgr = cv2.imread(str(sample["img_path"]))
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        # Load mask
        if sample["gt_path"] is not None:
            full_mask = self._load_mask(sample["gt_path"])
        else:
            full_mask = None

        # Resize
        image_rgb = cv2.resize(image_rgb, (self.img_size, self.img_size))
        if full_mask is not None:
            full_mask = cv2.resize(full_mask, (self.img_size, self.img_size),
                                   interpolation=cv2.INTER_NEAREST)

        # Augmentation (Albumentations)
        if self.transform is not None and full_mask is not None:
            transformed = self.transform(image=image_rgb, mask=full_mask)
            image_rgb = transformed["image"]
            full_mask = transformed["mask"]

        # Extract instances dari mask
        boxes = []
        masks = []
        labels = []

        if full_mask is not None and full_mask.sum() > 0:
            instances = self._extract_instances(full_mask)
            for inst_mask, bbox in instances:
                # Validasi bbox
                if bbox[2] <= bbox[0] or bbox[3] <= bbox[1]:
                    continue
                masks.append(inst_mask)
                boxes.append(bbox)
                labels.append(1)  # class 1 = target (OD atau HE)

        # Convert to tensors
        image_tensor = torch.from_numpy(image_rgb).permute(2, 0, 1).float() / 255.0

        if len(boxes) > 0:
            target = {
                "boxes": torch.as_tensor(boxes, dtype=torch.float32),
                "labels": torch.as_tensor(labels, dtype=torch.int64),
                "masks": torch.as_tensor(np.array(masks), dtype=torch.uint8),
                "image_id": idx,
            }
        else:
            # Negative sample (no instances)
            target = {
                "boxes": torch.zeros((0, 4), dtype=torch.float32),
                "labels": torch.zeros((0,), dtype=torch.int64),
                "masks": torch.zeros((0, self.img_size, self.img_size), dtype=torch.uint8),
                "image_id": idx,
            }

        return image_tensor, target

print("Dataset class OK")

## 4. Augmentasi (Albumentations)

In [30]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.1,
        scale_limit=0.2,
        rotate_limit=45,
        border_mode=cv2.BORDER_CONSTANT,
        p=0.5,
    ),
    A.ElasticTransform(alpha=120, sigma=12, p=0.2),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(var_limit=(10, 50), p=0.2),
])

# Tidak ada augmentasi untuk test
test_transform = None

print("Augmentasi OK")

Augmentasi OK


/var/folders/p3/0mc5zsd91bdcw4p37s0rxmf80000gn/T/ipykernel_83393/3791202423.py:14: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 50), p=0.2),


## 5. Model: Mask R-CNN + ResNet50

In [31]:
def get_mask_rcnn_model(num_classes=2, hidden_layer=256):
    """
    Load Mask R-CNN pre-trained pada COCO, ganti head untuk num_classes.
    num_classes = 2: background + 1 target class (OD atau HE)
    """
    # Load model pre-trained COCO
    model = maskrcnn_resnet50_fpn_v2(weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT)

    # Ganti box predictor
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Ganti mask predictor
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, hidden_layer, num_classes
    )

    return model

print("Model builder OK")

Model builder OK


## 6. Training Loop

In [32]:
def collate_fn(batch):
    """Custom collate karena target punya ukuran berbeda per gambar."""
    return tuple(zip(*batch))


def targets_to_device(targets, device):
    """Move target tensors ke device, skip non-tensor values."""
    result = []
    for t in targets:
        d = {}
        for k, v in t.items():
            if isinstance(v, torch.Tensor):
                d[k] = v.to(device)
            else:
                d[k] = v
        result.append(d)
    return result


def train_one_epoch(model, dataloader, optimizer, device, epoch):
    model.train()
    total_loss = 0
    n_batches = 0

    for images, targets in dataloader:
        images = [img.to(device) for img in images]
        targets = targets_to_device(targets, device)

        # Skip batch jika semua target kosong (Mask R-CNN butuh minimal 1 positive)
        has_positive = any(t["boxes"].shape[0] > 0 for t in targets)
        if not has_positive:
            continue

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        # Gradient clipping untuk stabilitas
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += losses.item()
        n_batches += 1

    avg_loss = total_loss / max(n_batches, 1)
    return avg_loss


def compute_iou_mask(pred_mask, gt_mask):
    """Compute IoU antara predicted mask dan ground truth mask (binary)."""
    intersection = (pred_mask & gt_mask).sum().float()
    union = (pred_mask | gt_mask).sum().float()
    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    return (intersection / union).item()


@torch.no_grad()
def evaluate(model, dataloader, device, conf_threshold=0.5):
    """
    Evaluasi model pada dataset.
    Untuk setiap gambar: gabungkan semua predicted masks, bandingkan dengan GT.
    Return: average IoU.
    """
    model.eval()
    ious = []

    for images, targets in dataloader:
        images = [img.to(device) for img in images]
        predictions = model(images)

        for pred, target in zip(predictions, targets):
            img_size = images[0].shape[-2:]

            # Gabungkan GT masks jadi satu binary mask
            if target["masks"].shape[0] > 0:
                gt_combined = target["masks"].any(dim=0).to(device)
            else:
                gt_combined = torch.zeros(img_size, dtype=torch.bool, device=device)

            # Gabungkan predicted masks (filter by confidence)
            if len(pred["scores"]) > 0:
                keep = pred["scores"] >= conf_threshold
                if keep.any():
                    pred_masks = pred["masks"][keep] > 0.5  # threshold mask probability
                    pred_combined = pred_masks.squeeze(1).any(dim=0)
                else:
                    pred_combined = torch.zeros(img_size, dtype=torch.bool, device=device)
            else:
                pred_combined = torch.zeros(img_size, dtype=torch.bool, device=device)

            iou = compute_iou_mask(pred_combined, gt_combined)
            ious.append(iou)

    avg_iou = np.mean(ious) if ious else 0.0
    return avg_iou, ious


print("Training & evaluation functions OK")

Training & evaluation functions OK


## 7. Training Function

In [33]:
def train_mask_rcnn(target_name, gt_config, num_epochs=20, batch_size=4,
                    lr=1e-4, img_size=512, patience=10):
    """
    Train Mask R-CNN untuk target tertentu (OD atau HE).

    Parameters:
        target_name: 'OD' atau 'HE'
        gt_config: dict dengan keys 'train', 'test', 'suffix'
        num_epochs: jumlah epoch
        batch_size: batch size
        lr: learning rate
        img_size: ukuran input image (resize)
        patience: early stopping patience
    """
    print(f"\n{'='*60}")
    print(f"  TRAINING MASK R-CNN untuk {target_name}")
    print(f"{'='*60}")
    print(f"  Epochs     : {num_epochs}")
    print(f"  Batch size : {batch_size}")
    print(f"  LR         : {lr}")
    print(f"  Image size : {img_size}x{img_size}")
    print(f"  Patience   : {patience}")
    print(f"  Device     : {DEVICE}")
    print()

    # === Dataset ===
    print("[Train set]")
    train_dataset = IDRiDSegDataset(
        img_dir=ORIG_TRAIN,
        gt_dir=gt_config["train"],
        gt_suffix=gt_config["suffix"],
        transform=train_transform,
        img_size=img_size,
    )
    print("[Test set]")
    test_dataset = IDRiDSegDataset(
        img_dir=ORIG_TEST,
        gt_dir=gt_config["test"],
        gt_suffix=gt_config["suffix"],
        transform=test_transform,
        img_size=img_size,
    )

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        collate_fn=collate_fn, num_workers=0,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_fn, num_workers=0,
    )

    # === Model ===
    model = get_mask_rcnn_model(num_classes=2)
    model.to(DEVICE)

    # === Optimizer & Scheduler ===
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=0.0005)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=5
    )

    # === Training ===
    best_iou = 0.0
    best_epoch = 0
    history = {"train_loss": [], "test_iou": [], "lr": []}
    save_dir = Path(f"runs/mask_rcnn_{target_name.lower()}")
    save_dir.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, num_epochs + 1):
        start = time.time()

        # Train
        train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE, epoch)

        # Evaluate
        test_iou, _ = evaluate(model, test_loader, DEVICE)

        # Scheduler step
        scheduler.step(test_iou)
        current_lr = optimizer.param_groups[0]["lr"]

        # History
        history["train_loss"].append(train_loss)
        history["test_iou"].append(test_iou)
        history["lr"].append(current_lr)

        elapsed = time.time() - start

        # Save best
        marker = ""
        if test_iou > best_iou:
            best_iou = test_iou
            best_epoch = epoch
            torch.save(model.state_dict(), save_dir / "best.pt")
            marker = " ★ BEST"

        print(
            f"  Epoch {epoch:3d}/{num_epochs} | "
            f"Loss: {train_loss:.4f} | "
            f"Test IoU: {test_iou:.4f} | "
            f"LR: {current_lr:.1e} | "
            f"{elapsed:.1f}s{marker}"
        )

        # Early stopping
        if epoch - best_epoch >= patience:
            print(f"\n  Early stopping at epoch {epoch} (best was epoch {best_epoch})")
            break

    # Save last & history
    torch.save(model.state_dict(), save_dir / "last.pt")
    with open(save_dir / "history.json", "w") as f:
        json.dump(history, f, indent=2)

    print(f"\n  ✓ Best IoU: {best_iou:.4f} (epoch {best_epoch})")
    print(f"  ✓ Model saved to {save_dir}")

    return model, history, best_iou

print("Training function OK")

Training function OK


---
## 8. Train Model OD (Optic Disc)

In [ ]:
od_model, od_history, od_best_iou = train_mask_rcnn(
    target_name="OD",
    gt_config=GT_PATHS["OD"],
    num_epochs=100,
    batch_size=4,
    lr=1e-4,
    img_size=512, 
    patience=15,
)


  TRAINING MASK R-CNN untuk OD
  Epochs     : 100
  Batch size : 4
  LR         : 0.0001
  Image size : 512x512
  Patience   : 15
  Device     : mps

[Train set]
  Dataset: 54 gambar, 54 dengan GT
[Test set]
  Dataset: 27 gambar, 27 dengan GT


Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Impacting Interactivity (0000000e:kIOGPUCommandBufferCallbackErrorImpactingInteractivity)
	<AGXG13XFamilyCommandBuffer: 0xb35259500>
    label = <none> 
    device = <AGXG13XDevice: 0xb4659c000>
        name = Apple M1 Pro 
    commandQueue = <AGXG13XFamilyCommandQueue: 0x104dbc380>
        label = <none> 
        device = <AGXG13XDevice: 0xb4659c000>
            name = Apple M1 Pro 
    retainedReferences = 1
Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Impacting Interactivity (0000000e:kIOGPUCommandBufferCallbackErrorImpactingInteractivity)
	<AGXG13XFamilyCommandBuffer: 0xb35259880>
    label = <none> 
    device = <AGXG13XDevice: 0xb4659c000>
        name = Apple M1 Pro 
    commandQueue = <AGXG13XFamilyCommandQueue: 0x104dbc380>
        la

## 9. Train Model HE (Hard Exudate)

In [ ]:
he_model, he_history, he_best_iou = train_mask_rcnn(
    target_name="HE",
    gt_config=GT_PATHS["HE"],
    num_epochs=200,
    batch_size=4,
    lr=5e-4,
    img_size=512,
    patience=30,
)

## 10. Visualisasi Training History

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, (name, history) in enumerate([("OD", od_history), ("HE", he_history)]):
    # Loss
    axes[i, 0].plot(history["train_loss"], "b-", label="Train Loss")
    axes[i, 0].set_title(f"{name} — Training Loss")
    axes[i, 0].set_xlabel("Epoch")
    axes[i, 0].set_ylabel("Loss")
    axes[i, 0].legend()
    axes[i, 0].grid(True, alpha=0.3)

    # IoU
    axes[i, 1].plot(history["test_iou"], "r-", label="Test IoU")
    axes[i, 1].set_title(f"{name} — Test IoU")
    axes[i, 1].set_xlabel("Epoch")
    axes[i, 1].set_ylabel("IoU")
    axes[i, 1].legend()
    axes[i, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("visualisasi_training_mask_rcnn.png", dpi=100, bbox_inches="tight")
plt.show()
print("Visualisasi disimpan ke visualisasi_training_mask_rcnn.png")

## 11. Evaluasi Detail pada Test Set

In [ ]:
def evaluate_detailed(model_path, target_name, gt_config, img_size=512):
    """Evaluasi detail per-gambar."""
    model = get_mask_rcnn_model(num_classes=2)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    model.to(DEVICE)
    model.eval()

    test_dataset = IDRiDSegDataset(
        img_dir=ORIG_TEST,
        gt_dir=gt_config["test"],
        gt_suffix=gt_config["suffix"],
        transform=None,
        img_size=img_size,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=1, shuffle=False,
        collate_fn=collate_fn, num_workers=0,
    )

    print(f"\n{'='*50}")
    print(f"  EVALUASI DETAIL: {target_name}")
    print(f"{'='*50}")

    all_ious = []
    with torch.no_grad():
        for idx, (images, targets) in enumerate(test_loader):
            images = [img.to(DEVICE) for img in images]
            preds = model(images)[0]
            target = targets[0]
            stem = test_dataset.samples[idx]["stem"]

            img_size_hw = images[0].shape[-2:]

            # GT mask
            if target["masks"].shape[0] > 0:
                gt_combined = target["masks"].any(dim=0).to(DEVICE)
            else:
                gt_combined = torch.zeros(img_size_hw, dtype=torch.bool, device=DEVICE)

            # Pred mask
            if len(preds["scores"]) > 0:
                keep = preds["scores"] >= 0.5
                if keep.any():
                    pred_masks = preds["masks"][keep] > 0.5
                    pred_combined = pred_masks.squeeze(1).any(dim=0)
                else:
                    pred_combined = torch.zeros(img_size_hw, dtype=torch.bool, device=DEVICE)
            else:
                pred_combined = torch.zeros(img_size_hw, dtype=torch.bool, device=DEVICE)

            iou = compute_iou_mask(pred_combined, gt_combined)
            all_ious.append(iou)
            print(f"  {stem}: IoU = {iou:.4f}")

    avg_iou = np.mean(all_ious)
    print(f"\n  Average IoU: {avg_iou:.4f}")
    return avg_iou, all_ious

# Evaluasi OD
od_avg_iou, od_ious = evaluate_detailed(
    model_path="runs/mask_rcnn_od/best.pt",
    target_name="Optic Disc",
    gt_config=GT_PATHS["OD"],
)

# Evaluasi HE
he_avg_iou, he_ious = evaluate_detailed(
    model_path="runs/mask_rcnn_he/best.pt",
    target_name="Hard Exudate",
    gt_config=GT_PATHS["HE"],
)

## 12. Visualisasi Prediksi

In [ ]:
def visualize_predictions(model_path, target_name, gt_config, samples=None, img_size=512):
    """Visualisasi prediksi vs ground truth."""
    model = get_mask_rcnn_model(num_classes=2)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    model.to(DEVICE)
    model.eval()

    test_dataset = IDRiDSegDataset(
        img_dir=ORIG_TEST,
        gt_dir=gt_config["test"],
        gt_suffix=gt_config["suffix"],
        transform=None,
        img_size=img_size,
    )

    if samples is None:
        # Ambil 6 gambar pertama yang punya GT
        samples = [
            i for i, s in enumerate(test_dataset.samples)
            if s["gt_path"] is not None
        ][:6]

    fig, axes = plt.subplots(len(samples), 3, figsize=(18, 5 * len(samples)))
    if len(samples) == 1:
        axes = axes[np.newaxis, :]

    with torch.no_grad():
        for row, idx in enumerate(samples):
            image_tensor, target = test_dataset[idx]
            stem = test_dataset.samples[idx]["stem"]

            # Predict
            pred = model([image_tensor.to(DEVICE)])[0]

            # Image
            img_np = image_tensor.permute(1, 2, 0).numpy()

            # GT mask
            if target["masks"].shape[0] > 0:
                gt_mask = target["masks"].any(dim=0).numpy().astype(np.uint8)
            else:
                gt_mask = np.zeros((img_size, img_size), dtype=np.uint8)

            # Pred mask
            if len(pred["scores"]) > 0:
                keep = pred["scores"].cpu() >= 0.5
                if keep.any():
                    pred_masks = pred["masks"][keep].cpu() > 0.5
                    pred_mask = pred_masks.squeeze(1).any(dim=0).numpy().astype(np.uint8)
                else:
                    pred_mask = np.zeros((img_size, img_size), dtype=np.uint8)
            else:
                pred_mask = np.zeros((img_size, img_size), dtype=np.uint8)

            # IoU
            iou = compute_iou_mask(
                torch.from_numpy(pred_mask).bool(),
                torch.from_numpy(gt_mask).bool(),
            )

            # Plot: Original | GT Overlay | Pred Overlay
            axes[row, 0].imshow(img_np)
            axes[row, 0].set_title(f"{stem} — Original")
            axes[row, 0].axis("off")

            gt_overlay = img_np.copy()
            gt_overlay[gt_mask > 0] = [0, 1, 0]  # green
            axes[row, 1].imshow(gt_overlay)
            axes[row, 1].set_title(f"{stem} — Ground Truth")
            axes[row, 1].axis("off")

            pred_overlay = img_np.copy()
            pred_overlay[pred_mask > 0] = [1, 0, 0]  # red
            axes[row, 2].imshow(pred_overlay)
            axes[row, 2].set_title(f"{stem} — Prediction (IoU: {iou:.3f})")
            axes[row, 2].axis("off")

    plt.suptitle(f"Mask R-CNN — {target_name} Segmentation", fontsize=16, y=1.01)
    plt.tight_layout()
    save_path = f"visualisasi_prediksi_{target_name.lower().replace(' ', '_')}.png"
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.show()
    print(f"Visualisasi disimpan ke {save_path}")

# Visualisasi OD
visualize_predictions(
    model_path="runs/mask_rcnn_od/best.pt",
    target_name="Optic Disc",
    gt_config=GT_PATHS["OD"],
)

# Visualisasi HE
visualize_predictions(
    model_path="runs/mask_rcnn_he/best.pt",
    target_name="Hard Exudate",
    gt_config=GT_PATHS["HE"],
)

## 13. Ringkasan Hasil

In [ ]:
print("="*60)
print("  RINGKASAN STAGE 1: Mask R-CNN + ResNet50")
print("="*60)
print()
print(f"  {'Model':<20} {'IoU Kami':>10} {'Baseline Paper':>15}")
print(f"  {'-'*20} {'-'*10} {'-'*15}")
print(f"  {'Optic Disc':<20} {od_avg_iou:>10.4f} {'0.843':>15}")
print(f"  {'Hard Exudate':<20} {he_avg_iou:>10.4f} {'0.993':>15}")
print()
print("  Weights tersimpan di:")
print("    - runs/mask_rcnn_od/best.pt")
print("    - runs/mask_rcnn_he/best.pt")
print()
print("  → Lanjut ke Stage 2: Preprocessing (Blackout + CLAHE)")
print("  → Kemudian Stage 3: Attention U-Net untuk Soft Exudate")